# Error analysis — v1 (DistilBERT transformer)

Test-set predictions from `ml/models/latest` (currently `v1`), compared
against ground truth. Goal: find systematic patterns in the mistakes
(not just the aggregate metrics in `ml/models/v1/metrics.json`), and
sanity-check whether "errors" are real model gaps or noisy source labels.

In [1]:
import pandas as pd
from pathlib import Path

from vibe_ml.models import load_classifier

DATA_DIR = Path("..") / "data" / "processed"
MODEL_DIR = Path("..") / "models" / "latest"

test_df = pd.read_csv(DATA_DIR / "test.csv")
model = load_classifier(MODEL_DIR.resolve())

preds = model.predict_batch_with_confidence(test_df["text"].tolist())
test_df["pred_mood"] = [mood for mood, _ in preds]
test_df["confidence"] = [conf for _, conf in preds]
test_df["correct"] = test_df["mood"] == test_df["pred_mood"]

print(f"accuracy: {test_df['correct'].mean():.3f}  (sanity check vs metrics.json)")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

accuracy: 0.684  (sanity check vs metrics.json)


## Top confusion pairs

Every (true, predicted) mismatch pair, ranked by how often it happens.

In [2]:
errors = test_df[~test_df["correct"]]
confusion_pairs = errors.groupby(["mood", "pred_mood"]).size().sort_values(ascending=False)
confusion_pairs.head(15)

mood      pred_mood
neutral   angry        245
          happy        154
happy     neutral      122
angry     neutral       88
neutral   sad           79
          romantic      76
happy     romantic      68
          excited       46
          angry         40
angry     sad           34
neutral   excited       33
angry     happy         33
sad       angry         29
romantic  neutral       27
          happy         27
dtype: int64

## Confidently wrong

Errors are expected — but a *confident* error (model very sure, still
wrong) is more concerning than a low-confidence one near a genuine
decision boundary. Compare confidence distributions, then look at the
worst offenders.

In [3]:
print("mean confidence, correct:  ", test_df.loc[test_df["correct"], "confidence"].mean())
print("mean confidence, incorrect:", test_df.loc[~test_df["correct"], "confidence"].mean())

confidently_wrong = errors[errors["confidence"] > 0.9].sort_values("confidence", ascending=False)
print(f"\n{len(confidently_wrong)} errors with confidence > 0.9 (out of {len(errors)} total errors)")
confidently_wrong[["text", "mood", "pred_mood", "source_label", "confidence"]].head(15)

mean confidence, correct:   0.8382107340502091
mean confidence, incorrect: 0.7014697271448965

237 errors with confidence > 0.9 (out of 1310 total errors)


,text,mood,pred_mood,source_label,confidence
96,What happened to one of your star players?? i ...,neutral,anxious,neutral,0.992707
2513,Cant wait to draft my man early to mid second ...,neutral,excited,neutral,0.992031
2630,Have you ever consider knocking three times an...,angry,anxious,annoyance,0.990264
1104,Asides from the max number of choices each sha...,neutral,anxious,neutral,0.990207
2814,"Wow, that's crazy. Ever tried DMT?",happy,excited,admiration,0.989786
3732,Omg yes!,happy,excited,approval,0.989359
2115,I'm really excited for [NAME]. He seems to be ...,happy,excited,optimism,0.989345
1861,Noooo no more NEXTs anymore! They are obviousl...,angry,excited,disapproval,0.986935
3394,but it's [NAME] saying it so sorta disappointing,neutral,sad,neutral,0.986838
3175,Then I’m sorry but this game really isn’t for ...,angry,sad,disapproval,0.986419


## Reading the confidently-wrong examples

Is the model actually wrong, or is the *GoEmotions source label* wrong /
genuinely ambiguous (crowd-sourced labels are noisy, and single-label
mapping forces a choice on text that may carry more than one emotion)?

In [4]:
for _, row in confidently_wrong.head(10).iterrows():
    print(f"[{row['source_label']} -> {row['mood']}]  predicted={row['pred_mood']} (conf={row['confidence']:.2f})")
    print(f"  {row['text']!r}\n")

[neutral -> neutral]  predicted=anxious (conf=0.99)
  "What happened to one of your star players?? i saw the headline 'freak injury' or someting like that.. but i was too scared to see someting gruesome.."

[neutral -> neutral]  predicted=excited (conf=0.99)
  "Cant wait to draft my man early to mid second round next year. If I'm drafting late first I might really go wr/wr"

[annoyance -> angry]  predicted=anxious (conf=0.99)
  "Have you ever consider knocking three times and that hide and seek/stalker vibe is terrifying. It's the most disturbing thing I can link you my AR 15."

[neutral -> neutral]  predicted=anxious (conf=0.99)
  'Asides from the max number of choices each share, probably being shamed on the internet'

[admiration -> happy]  predicted=excited (conf=0.99)
  "Wow, that's crazy. Ever tried DMT? "

[approval -> happy]  predicted=excited (conf=0.99)
  'Omg yes!'

[optimism -> happy]  predicted=excited (conf=0.99)
  "I'm really excited for [NAME]. He seems to be continuall

## Revisiting the earlier miss

Manual testing earlier found `predict_vibe("I've had a terrible day, I
just want to lie down and forget everything.")` predicts `anxious`, not
the more intuitive `sad` (confidence 0.85). Check the sad/anxious
boundary specifically in the test set.

In [5]:
sad_as_anxious = test_df[(test_df["mood"] == "sad") & (test_df["pred_mood"] == "anxious")]
print(f"{len(sad_as_anxious)} true-sad rows predicted as anxious")
sad_as_anxious[["text", "source_label", "confidence"]].head(10)

5 true-sad rows predicted as anxious


,text,source_label,confidence
427,It’s a terrible thing when someone who doesn’t...,disappointment,0.985972
508,Not at all. It's great of course for financial...,sadness,0.978240
928,Our soil needs the help! That would be a reall...,disappointment,0.678084
3026,"Yes of mine worry, no one liked mine either.",disappointment,0.624704
3965,I feel sorry for this little girl. She's legit...,remorse,0.976261


## Summary

See `docs/ml-plan.md` ("Error analysis" section) for the write-up — kept
there rather than duplicated here so there's one place findings live.